**Problem 1 - Bot-aware sessionization + week-over-week retention**

In [1]:
import sys
print(sys.executable)

C:\Users\flood\anaconda3\envs\ml_env\python.exe


In [2]:
import os
from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

spark = (
    SparkSession.builder
    .appName("HW2")
    .master("local[*]")
    .getOrCreate()
)

spark

In [3]:
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import StringType
import re

In [4]:
event = spark.read.json("data/p1_raw_events.jsonl")
event = event.withColumn(
    "event_ts",
    F.to_timestamp("event_ts")
).withColumn(
    "ingested_at",
    F.to_timestamp("ingested_at")
)
event.printSchema()
event.show(5, truncate=False)

root
 |-- event_id: string (nullable = true)
 |-- event_name: string (nullable = true)
 |-- event_ts: timestamp (nullable = true)
 |-- ingested_at: timestamp (nullable = true)
 |-- page: string (nullable = true)
 |-- user_agent: string (nullable = true)
 |-- user_id: string (nullable = true)

+--------+----------+-------------------+-------------------+----------------------+------------------------------------------------------------+-------+
|event_id|event_name|event_ts           |ingested_at        |page                  |user_agent                                                  |user_id|
+--------+----------+-------------------+-------------------+----------------------+------------------------------------------------------------+-------+
|E200000 |page_view |2026-02-24 19:06:43|2026-02-24 19:10:41|/home                 |Mozilla/5.0 (X11; Linux x86_64) Gecko/20100101 Firefox/122.0|U00000 |
|E200001 |click     |2026-02-24 19:11:11|2026-02-24 19:13:54|/help                 |Mozill

In [5]:
@F.udf(StringType())
def classify(user_agent):

    if user_agent is None:
        return "OTHER"

    s = str(user_agent).strip()

    if s == "":
        return "OTHER"

    s_lower = s.lower()

    # words that should cancel bot detection
    if any(x in s_lower for x in ["robotics", "reboot", "botanist"]):
        pass
    else:
        if re.search(r"\bbot\b", s_lower):
            return "BOT"
        if re.search(r"\bcrawler\b", s_lower):
            return "BOT"
        if re.search(r"\bspider\b", s_lower):
            return "BOT"
        if re.search(r"\bslurp\b", s_lower):
            return "BOT"
        if s_lower.startswith("curl/"):
            return "BOT"
        if s_lower.startswith("wget/"):
            return "BOT"
        if s_lower.startswith("python-requests/"):
            return "BOT"
        if "headlesschrome" in s_lower:
            return "BOT"

    if "ipad" in s_lower:
        return "TABLET"

    if "android" in s_lower and "mobile" not in s_lower:
        return "TABLET"

    if "iphone" in s_lower:
        return "MOBILE"

    if "android" in s_lower and "mobile" in s_lower:
        return "MOBILE"

    if any(x in s_lower for x in ["windows nt", "macintosh", "x11", "linux"]):
        return "DESKTOP"

    return "OTHER"


    

In [6]:
dedup_w = (
    Window
    .partitionBy("event_id")
    .orderBy(
        F.col("ingested_at").desc_nulls_last(),
        F.col("event_ts").desc_nulls_last(),
        F.col("event_name").asc_nulls_last(),
        F.col("page").asc_nulls_last()
    )
)

event_dedup = (
    event
    .withColumn("rn_dedup", F.row_number().over(dedup_w))
    .filter(F.col("rn_dedup") == 1)
    .drop("rn_dedup")
)

In [7]:
events_with_ua = (
    event_dedup
    .withColumn("ua_type", classify(F.col("user_agent")))
)

events_nonbot = events_with_ua.filter(F.col("ua_type") != "BOT")

events_local = (
    events_nonbot
    .withColumn("local_ts", F.from_utc_timestamp(F.col("event_ts"), "America/New_York"))
    .withColumn("local_date", F.to_date("local_ts"))
)

In [8]:
session_order_w = (
    Window
    .partitionBy("user_id")
    .orderBy(
        F.col("event_ts").asc_nulls_last(),
        F.col("ingested_at").asc_nulls_last(),
        F.col("event_id").asc_nulls_last(),
        F.col("event_name").asc_nulls_last(),
        F.col("page").asc_nulls_last()
    )
)

In [9]:
events_sessions = (
    events_local
    .withColumn("prev_event_ts", F.lag("event_ts").over(session_order_w))
    .withColumn("prev_local_date", F.lag("local_date").over(session_order_w))
    .withColumn(
        "gap_minutes",
        (
            F.unix_timestamp("event_ts") - 
        F.unix_timestamp("prev_event_ts")
        ) / 60
    )
    .withColumn(
        "is_session_start",
        F.when(F.col("prev_event_ts").isNull(), F.lit(1))
         .when(F.col("gap_minutes") > F.lit(25), F.lit(1))
         .when(F.col("local_date") != F.col("prev_local_date"), F.lit(1))
         .otherwise(F.lit(0))
    )
    .withColumn(
        "session_index",
        F.sum("is_session_start").over(
            session_order_w.rowsBetween(Window.unboundedPreceding, Window.currentRow)
        )
    )
    .withColumn(
        "session_id",
        F.concat(
            F.col("user_id"),
            F.lit("#"),
            F.lpad(F.col("session_index").cast("string"), 6, "0")
        )
    )
    .drop("prev_local_date")
)

In [10]:
events_sessions = (
    events_sessions
    .withColumn(
        "week_start_date",
        F.date_sub(F.next_day(F.to_date("local_ts"), "Mon"),7)
    )
)

In [11]:
weekly_base = (
    events_sessions
    .groupBy("user_id", "week_start_date")
    .agg(
        F.countDistinct("session_id").alias("sessions"),
        F.count(F.lit(1)).alias("events"),
        F.countDistinct(F.to_date("local_ts")).alias("active_days_in_week")
    )
)

In [12]:
first_ua_w = (
    Window
    .partitionBy("user_id", "week_start_date")
    .orderBy(
        F.col("event_ts").asc_nulls_last(),
        F.col("event_id").asc_nulls_last()
    )
)

weekly_first_ua = (
    events_sessions
    .withColumn("rn_first_week", F.row_number().over(first_ua_w))
    .filter(F.col("rn_first_week") == 1)
    .select(
        "user_id",
        "week_start_date",
        F.col("ua_type").alias("first_ua_type_in_week")
    )
)

In [13]:
retention_w = (
    Window
    .partitionBy("user_id")
    .orderBy("week_start_date")
)

weekly_retention = (
    weekly_base
    .withColumn("next_week_start", F.lead("week_start_date").over(retention_w))
    .withColumn(
        "returning_next_week",
        F.when(
            F.datediff(F.col("next_week_start"), F.col("week_start_date")) == 7,
            F.lit(1)
        ).otherwise(F.lit(0))
    )
    .drop("next_week_start")
)

In [14]:
user_week_metrics = (
    weekly_retention
    .join(
        weekly_first_ua,
        on=["user_id", "week_start_date"],
        how="left"
    )
    .select(
        "user_id",
        "week_start_date",
        "sessions",
        "events",
        "first_ua_type_in_week",
        "active_days_in_week",
        "returning_next_week"
    )
)

In [15]:
events_sessions.orderBy(
    "user_id",
    "event_ts",
    "ingested_at",
    "event_id"
).show(20, truncate=False)

+--------+----------+-------------------+-------------------+---------------+---------------------------------------------------------------------------------------------+-------+-------+-------------------+----------+-------------------+-------------------+----------------+-------------+-------------+---------------+
|event_id|event_name|event_ts           |ingested_at        |page           |user_agent                                                                                   |user_id|ua_type|local_ts           |local_date|prev_event_ts      |gap_minutes        |is_session_start|session_index|session_id   |week_start_date|
+--------+----------+-------------------+-------------------+---------------+---------------------------------------------------------------------------------------------+-------+-------+-------------------+----------+-------------------+-------------------+----------------+-------------+-------------+---------------+
|E200004 |click     |2026-02-24 11:42:17

In [16]:
user_week_metrics.orderBy(
    "user_id",
    "week_start_date"
).show(20, truncate=False)

+-------+---------------+--------+------+---------------------+-------------------+-------------------+
|user_id|week_start_date|sessions|events|first_ua_type_in_week|active_days_in_week|returning_next_week|
+-------+---------------+--------+------+---------------------+-------------------+-------------------+
|U00000 |2026-02-23     |2       |8     |DESKTOP              |1                  |0                  |
|U00001 |2026-02-16     |1       |4     |MOBILE               |1                  |0                  |
|U00002 |2026-02-09     |1       |3     |MOBILE               |1                  |1                  |
|U00002 |2026-02-16     |1       |5     |DESKTOP              |1                  |1                  |
|U00002 |2026-02-23     |1       |2     |DESKTOP              |1                  |0                  |
|U00003 |2026-02-16     |1       |5     |DESKTOP              |1                  |0                  |
|U00004 |2026-02-02     |2       |6     |DESKTOP              |2

In [17]:
events_sessions.printSchema()

root
 |-- event_id: string (nullable = true)
 |-- event_name: string (nullable = true)
 |-- event_ts: timestamp (nullable = true)
 |-- ingested_at: timestamp (nullable = true)
 |-- page: string (nullable = true)
 |-- user_agent: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- ua_type: string (nullable = true)
 |-- local_ts: timestamp (nullable = true)
 |-- local_date: date (nullable = true)
 |-- prev_event_ts: timestamp (nullable = true)
 |-- gap_minutes: double (nullable = true)
 |-- is_session_start: integer (nullable = false)
 |-- session_index: long (nullable = true)
 |-- session_id: string (nullable = true)
 |-- week_start_date: date (nullable = true)



In [18]:
user_week_metrics.printSchema()

root
 |-- user_id: string (nullable = true)
 |-- week_start_date: date (nullable = true)
 |-- sessions: long (nullable = false)
 |-- events: long (nullable = false)
 |-- first_ua_type_in_week: string (nullable = true)
 |-- active_days_in_week: long (nullable = false)
 |-- returning_next_week: integer (nullable = false)



**Problem 2 -  Prorated February billing with pauses**

In [19]:
memberships = spark.read.json("data/p2_memberships.jsonl")
plan_dim = spark.read.option("header", True).csv("data/p2_plan_dim.csv", inferSchema=True)

In [20]:

from pyspark.sql.types import DecimalType
import hashlib

memberships = (
    memberships
    .withColumn("start_date", F.to_date("start_date"))
    .withColumn("end_date", F.to_date("end_date"))
)

plan_dim = (
    plan_dim
    .withColumn("monthly_fee", F.col("monthly_fee").cast(DecimalType(10, 2)))
)

memberships.printSchema()
plan_dim.printSchema()

root
 |-- end_date: date (nullable = true)
 |-- member_id: string (nullable = true)
 |-- pause_ranges: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- pause_end: string (nullable = true)
 |    |    |-- pause_start: string (nullable = true)
 |-- plan_id: string (nullable = true)
 |-- start_date: date (nullable = true)

root
 |-- plan_id: string (nullable = true)
 |-- monthly_fee: decimal(10,2) (nullable = true)



In [21]:
report_start = F.lit("2026-02-01").cast("date")
report_end_exclusive = F.lit("2026-03-01").cast("date")
days_in_month_lit = F.lit(28)

In [22]:
@F.udf(StringType())
def billing_cohort(member_id):
    if member_id is None:
        return "D"
    s = str(member_id)
    h = hashlib.md5(s.encode("utf-8")).hexdigest()
    value = int(h[:2], 16) % 4
    mapping = {0: "A", 1: "B", 2: "C", 3: "D"}
    return mapping[value]

In [23]:
memberships_active = (
    memberships
    .withColumn("active_start", F.greatest(F.col("start_date"), report_start))
    .withColumn(
        "membership_end_exclusive",
        F.when(
            F.col("end_date").isNotNull(),
            F.date_add(F.col("end_date"), 1)
        ).otherwise(report_end_exclusive)
    )
    .withColumn(
        "active_end_exclusive",
        F.least(F.col("membership_end_exclusive"), report_end_exclusive)
    )
    .filter(F.col("active_start") < F.col("active_end_exclusive"))
    .withColumn("cohort", billing_cohort(F.col("member_id")))
)

In [24]:
member_days_base = (
    memberships_active
    .withColumn(
        "day",
        F.explode(
            F.sequence(
                F.col("active_start"),
                F.date_sub(F.col("active_end_exclusive"), 1)
            )
        )
    )
    .select(
        "member_id",
        "plan_id",
        "day",
        "cohort",
        "pause_ranges"
    )
)

In [25]:
pause_days = (
    memberships_active
    .select("member_id", "plan_id", "pause_ranges")
    .withColumn("pause_range", F.explode_outer("pause_ranges"))
    .withColumn("pause_start", F.to_date(F.col("pause_range.pause_start")))
    .withColumn("pause_end_raw", F.to_date(F.col("pause_range.pause_end")))
    .withColumn(
        "pause_end_exclusive",
        F.when(F.col("pause_end_raw").isNull(), report_end_exclusive)
         .otherwise(F.col("pause_end_raw"))
    )
    .filter(F.col("pause_start").isNotNull())
    .filter(
        F.col("pause_end_raw").isNull() | (F.col("pause_end_exclusive") > F.col("pause_start"))
    )
    .withColumn("pause_start_clamped", F.greatest(F.col("pause_start"), report_start))
    .withColumn("pause_end_clamped", F.least(F.col("pause_end_exclusive"), report_end_exclusive))
    .filter(F.col("pause_start_clamped") < F.col("pause_end_clamped"))
    .withColumn(
        "day",
        F.explode(
            F.sequence(
                F.col("pause_start_clamped"),
                F.date_sub(F.col("pause_end_clamped"), 1)
            )
        )
    )
    .select("member_id", "plan_id", "day")
    .distinct()
)

In [26]:
member_days = (
    member_days_base
    .join(
        pause_days.withColumn("is_paused", F.lit(1)),
        on=["member_id", "plan_id", "day"],
        how="left"
    )
    .withColumn(
        "is_billable",
        F.when(F.col("is_paused") == 1, F.lit(0)).otherwise(F.lit(1))
    )
    .select(
        "member_id",
        "plan_id",
        "day",
        "cohort",
        "is_billable"
    )
)

In [27]:
member_feb_billing = (
    member_days
    .groupBy("member_id", "plan_id", "cohort")
    .agg(
        F.sum("is_billable").alias("billable_days")
    )
    .join(plan_dim, on="plan_id", how="left")
    .withColumn("days_in_month", days_in_month_lit.cast("int"))
    .withColumn(
        "prorated_fee",
        (
            F.col("monthly_fee").cast(DecimalType(18, 4)) *
            F.col("billable_days").cast(DecimalType(18, 4)) /
            F.col("days_in_month").cast(DecimalType(18, 4))
        ).cast(DecimalType(18, 2))
    )
    .select(
        "member_id",
        "plan_id",
        "cohort",
        "billable_days",
        "prorated_fee"
    )
)

In [28]:
daily_cohort_metrics = (
    member_days
    .groupBy("day", "plan_id", "cohort")
    .agg(
        F.countDistinct("member_id").alias("active_members"),
        F.countDistinct(F.when(F.col("is_billable") == 1, F.col("member_id"))).alias("billable_members"),
        F.countDistinct(F.when(F.col("is_billable") == 0, F.col("member_id"))).alias("paused_members")
    )
    .orderBy("day", "plan_id", "cohort")
)

In [29]:
print("member_days")

member_days.orderBy(
    "member_id",
    "plan_id",
    "day"
).show(50, truncate=False)

member_days
+---------+-------+----------+------+-----------+
|member_id|plan_id|day       |cohort|is_billable|
+---------+-------+----------+------+-----------+
|NULL     |basic  |2026-02-03|D     |1          |
|NULL     |basic  |2026-02-04|D     |1          |
|NULL     |basic  |2026-02-05|D     |1          |
|NULL     |basic  |2026-02-06|D     |1          |
|NULL     |basic  |2026-02-07|D     |1          |
|NULL     |basic  |2026-02-08|D     |1          |
|NULL     |basic  |2026-02-09|D     |1          |
|NULL     |basic  |2026-02-10|D     |1          |
|NULL     |basic  |2026-02-11|D     |1          |
|NULL     |basic  |2026-02-12|D     |1          |
|NULL     |basic  |2026-02-12|D     |1          |
|NULL     |basic  |2026-02-13|D     |1          |
|NULL     |basic  |2026-02-13|D     |1          |
|NULL     |basic  |2026-02-14|D     |1          |
|NULL     |basic  |2026-02-14|D     |1          |
|NULL     |basic  |2026-02-15|D     |1          |
|NULL     |basic  |2026-02-15|D     |1

In [30]:
print(" member_feb_billing")

member_feb_billing.orderBy(
    "member_id",
    "plan_id"
).show(20, truncate=False)

 member_feb_billing
+---------+----------+------+-------------+------------+
|member_id|plan_id   |cohort|billable_days|prorated_fee|
+---------+----------+------+-------------+------------+
|NULL     |basic     |D     |47           |134.29      |
|NULL     |pro       |D     |51           |218.57      |
|M000001  |pro       |B     |18           |77.14       |
|M000002  |basic     |D     |28           |80.00       |
|M000003  |basic     |B     |17           |48.57       |
|M000004  |basic     |B     |26           |74.29       |
|M000005  |basic     |B     |28           |80.00       |
|M000006  |pro       |A     |20           |85.71       |
|M000008  |basic     |A     |28           |80.00       |
|M000009  |basic     |A     |0            |0.00        |
|M000010  |pro       |B     |28           |120.00      |
|M000011  |pro       |A     |28           |120.00      |
|M000012  |basic     |B     |14           |40.00       |
|M000013  |enterprise|C     |28           |300.00      |
|M000014  |

In [31]:
print("daily_cohort_metrics")

daily_cohort_metrics.orderBy(
    "day",
    "plan_id",
    "cohort"
).show(20, truncate=False)

daily_cohort_metrics
+----------+----------+------+--------------+----------------+--------------+
|day       |plan_id   |cohort|active_members|billable_members|paused_members|
+----------+----------+------+--------------+----------------+--------------+
|2026-02-01|basic     |A     |5086          |4685            |401           |
|2026-02-01|basic     |B     |5152          |4785            |367           |
|2026-02-01|basic     |C     |5150          |4766            |384           |
|2026-02-01|basic     |D     |5227          |4837            |390           |
|2026-02-01|enterprise|A     |684           |637             |47            |
|2026-02-01|enterprise|B     |630           |581             |49            |
|2026-02-01|enterprise|C     |689           |640             |49            |
|2026-02-01|enterprise|D     |652           |609             |43            |
|2026-02-01|pro       |A     |2509          |2303            |206           |
|2026-02-01|pro       |B     |2543         

**Problem 3 -   Canonical baskets + signature statistics**

In [32]:
order_lines = spark.read.json("data/p3_order_lines.jsonl")

In [33]:
order_lines = (
    order_lines
    .withColumn("order_ts", F.to_timestamp("order_ts"))
    .withColumn(
        "unit_price",
        F.coalesce(
            F.col("unit_price").cast(DecimalType(10,2)),
            F.lit(0).cast(DecimalType(10,2))
        )
    )
)

In [34]:
@F.udf(StringType())
def normalize_sku(sku_raw):
    if sku_raw is None:
        return None
    
    s = str(sku_raw).strip()
    if s == "":
        return None

    s = s.upper()

    if s.startswith("SKU:"):
        s = s[4:]

    s = re.sub(r"[ .-]+", "-", s)

    if "#" in s and re.search(r"#\d{2}$", s):
        s = re.sub(r"#\d{2}$", "", s)

    if s == "":
        return None

    return s

In [35]:
order_lines_enriched = (
    order_lines
    .withColumn("sku", normalize_sku(F.col("sku_raw")))
    .withColumn("order_date_ny",
        F.to_date(F.from_utc_timestamp(F.col("order_ts"), "America/New_York"))
    )
    .withColumn(
        "line_revenue",
        (F.col("qty") * F.col("unit_price")).cast(DecimalType(18,2))
    )
)

In [36]:
order_line_arrays = (
    order_lines_enriched
    .groupBy("order_id")
    .agg(
        F.collect_list(
            F.struct(
                F.col("line_number").alias("sort_line_number"),
                F.when(F.col("sku").isNull(), F.lit(1)).otherwise(F.lit(0)).alias("sort_sku_null"),
                F.col("sku").alias("sort_sku"),
                (-F.col("line_revenue")).alias("sort_neg_line_revenue"),
                F.struct(
                    F.col("line_number").alias("line_number"),
                    F.col("sku").alias("sku"),
                    F.col("qty").alias("qty"),
                    F.col("line_revenue").alias("line_revenue")
                ).alias("line_struct")
            )
        ).alias("line_sort_array")
    )
    .withColumn("line_sort_array", F.array_sort("line_sort_array"))
    .withColumn(
        "lines",
        F.expr("transform(line_sort_array, x -> x.line_struct)")
    )
    .drop("line_sort_array")
)

In [37]:
order_base = (
    order_lines_enriched
    .groupBy("order_id")
    .agg(
        F.min("customer_id").alias("customer_id"),
        F.min("order_date_ny").alias("order_date_ny"),
        F.sum("line_revenue").cast(DecimalType(18, 2)).alias("gross_revenue"),
        F.count(F.lit(1)).alias("basket_size"),
        F.countDistinct("sku").alias("distinct_skus")
    )
)

In [38]:
signature_parts = (
    order_lines_enriched
    .filter(F.col("sku").isNotNull())
    .groupBy("order_id", "sku")
    .agg(F.sum("qty").alias("net_qty"))
    .filter(F.col("net_qty") != 0)
    .withColumn(
        "sku_net_str",
        F.concat(F.col("sku"), F.lit("="), F.col("net_qty").cast("string"))
    )
)

basket_signatures = (
    signature_parts
    .groupBy("order_id")
    .agg(
        F.array_sort(F.collect_list("sku_net_str")).alias("sig_parts")
    )
    .withColumn(
        "basket_signature",
        F.concat_ws("|", F.col("sig_parts"))
    )
    .select("order_id", "basket_signature")
)

In [39]:
all_orders = order_lines_enriched.select("order_id").distinct()

basket_signatures_full = (
    all_orders
    .join(basket_signatures, on="order_id", how="left")
    .withColumn("basket_signature", F.coalesce(F.col("basket_signature"), F.lit("")))
)

In [40]:
order_baskets = (
    order_base
    .join(order_line_arrays, on="order_id", how="inner")
    .join(basket_signatures_full, on="order_id", how="left")
    .select(
        "order_id",
        "customer_id",
        "order_date_ny",
        "gross_revenue",
        "lines",
        "distinct_skus",
        "basket_size",
        "basket_signature"
    )
)

In [41]:
signature_base_stats = (
    order_baskets
    .groupBy("basket_signature")
    .agg(
        F.count(F.lit(1)).alias("orders"),
        F.countDistinct("customer_id").alias("unique_customers"),
        F.avg("gross_revenue").cast(DecimalType(18, 2)).alias("avg_gross_revenue")
    )
)

In [42]:
signature_sku_totals = (
    order_lines_enriched
    .filter(F.col("sku").isNotNull())
    .groupBy("order_id", "sku")
    .agg(F.sum("qty").alias("net_qty_per_order_sku"))
    .join(basket_signatures_full, on="order_id", how="left")
    .groupBy("basket_signature", "sku")
    .agg(F.sum("net_qty_per_order_sku").alias("total_net_qty"))
)

signature_max_qty = (
    signature_sku_totals
    .groupBy("basket_signature")
    .agg(F.max("total_net_qty").alias("max_total_net_qty"))
)

signature_top_sku = (
    signature_sku_totals
    .join(
        signature_max_qty,
        on="basket_signature",
        how="inner"
    )
    .filter(F.col("total_net_qty") == F.col("max_total_net_qty"))
    .groupBy("basket_signature")
    .agg(F.min("sku").alias("top_sku"))
)

In [43]:
signature_stats = (
    signature_base_stats
    .join(signature_top_sku, on="basket_signature", how="left")
    .select(
        "basket_signature",
        "orders",
        "unique_customers",
        "avg_gross_revenue",
        "top_sku"
    )
    .orderBy(F.col("orders").desc(), F.col("basket_signature").asc())
)

In [44]:
print("=== order_baskets (sample) ===")
order_baskets.orderBy("order_id").show(20, truncate=False)

print("=== signature_stats (sample) ===")
signature_stats.show(20, truncate=False)

=== order_baskets (sample) ===
+--------+-----------+-------------+-------------+------------------------------------------------------------------------------------------------------------------------------+-------------+-----------+------------------------------------+
|order_id|customer_id|order_date_ny|gross_revenue|lines                                                                                                                         |distinct_skus|basket_size|basket_signature                    |
+--------+-----------+-------------+-------------+------------------------------------------------------------------------------------------------------------------------------+-------------+-----------+------------------------------------+
|O300000 |C03088     |2026-02-21   |220.65       |[{1, JT-92, 1, 35.56}, {1, JT-92, -1, -35.56}, {2, JT-92, 3, 399.18}, {3, JT-92, -1, -178.53}]                                |1            |4          |JT-92=2                             |
|O300

In [ ]:
!git add .
!git commit -m "P3: canonical order baskets, SKU normalization UDF, basket signatures, and signature-level statistics"
!git push -u origin main